# panvram quickstart - the HPRC cohort (558 assemblies) resident on one GPU

Downloads everything it needs (no Google Drive): the panvram package, T2T-CHM13v2.0 from NCBI (md5 checked) and the
refrel3 v1 archives as one tar per dataset shipped in parts (every part against `PARTS.sha256`, the joined tar against its `.sha256`, then every archive against `MANIFEST.tsv`). Then opens the
cohort on the GPU, samples windows, fetches regions and decodes every assembly in full, checking each FASTA XXH3
against the manifest (no source FASTA needed).

**Before running:** set `DATASET_BASE_URL` (the dataset record - placeholder until the DOI exists - or a local / Drive directory holding the same files) and `PANVRAM_SOURCE` (the package release tag, or a local tarball) - the package source. Needs a CUDA GPU of sm_80 or newer; q4k needs ~37 GB of disk at the peak
(parts 12.3 GB + the joined tar 12.3 GB + archives 12.3 GB; parts and tar are removed as soon as they are verified) + 3.2 GB for T2T,
and ~16.7 GB of GPU memory in the default form (11.7 GB with both compact forms).

Data use: HPRC data are public domain; do not sell them, do not attempt participant re-identification, acknowledge
"the Human Pangenome Reference Consortium (BioProject ID: PRJNA730823) and its funder, the National Human Genome
Research Institute (NHGRI)" (see DATASET.md).

In [ ]:
# 1. Settings
import os
DATASET_BASE_URL = "https://zenodo.org/records/REPLACE-RECORD-ID/files"   # placeholder until the DOI exists; a local/Drive directory path also works
SUFFIX = "?download=1" if DATASET_BASE_URL.startswith("http") else ""
PANVRAM_SOURCE = "https://github.com/yasha1971-coder/panvram/archive/refs/tags/v1.0.0.tar.gz"  # the package (tag v1.0.0); a local/Drive tarball path also works
DATASET = "q4k"            # q4k or q16k
FORMS = "0,0 1,1"          # resident forms "packed_reference,compact_blocks"
W = "/content/panvram_run"
os.environ.update(dict(DATASET_BASE_URL=DATASET_BASE_URL, SUFFIX=SUFFIX, PANVRAM_SOURCE=PANVRAM_SOURCE, DATASET=DATASET, FORMS=FORMS, W=W))



In [ ]:
%%bash
# 2. panvram: download the package and build the CUDA path
set -euo pipefail
mkdir -p $W && cd $W
[ -d panvram ] || { if [[ "$PANVRAM_SOURCE" == http* ]]; then curl -fL --retry 3 -o panvram.tar.gz "$PANVRAM_SOURCE"; else cp "$PANVRAM_SOURCE" panvram.tar.gz; fi; mkdir -p panvram && tar -xzf panvram.tar.gz -C panvram --strip-components=1; }
cd panvram && pip install -q -e . && python -c "import panvram; assert panvram.with_cuda, 'no CUDA path'; print('panvram', panvram.__file__)"


In [ ]:
%%bash
# 3. Reference: T2T-CHM13v2.0 from NCBI, md5 of the unpacked FASTA checked
set -euo pipefail
mkdir -p $W/cohort && cd $W
if [ ! -s t2t.fa ]; then
  curl -fL --retry 3 -o t2t.fa.gz https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz
  gunzip -c t2t.fa.gz > t2t.fa && rm -f t2t.fa.gz
fi
echo "cd1e52ce400c027ed0b7ab4b9d613f5a  t2t.fa" | md5sum -c -
ln -sf $W/t2t.fa $W/cohort/t2t.fa


In [ ]:
%%bash
# 4. Archives: the dataset tar is shipped as parts of 2 000 MiB. Parts -> SHA-256 of every part (PARTS.sha256) -> cat in
#    name order -> SHA-256 of the whole tar (.sha256) -> unpacked -> SHA-256 of every archive against MANIFEST.tsv
set -euo pipefail
cd $W
t=cohort558_$DATASET.tar
get() { [ -s "$1" ] && return 0; if [[ "$DATASET_BASE_URL" == http* ]]; then curl -fL --retry 3 -o "$1" "$DATASET_BASE_URL/$1$SUFFIX"; else cp "$DATASET_BASE_URL/$1" "$1"; fi; }
get PARTS.sha256; get $t.sha256
parts=$(awk '{print $2}' PARTS.sha256 | grep "^$t.part" | sort)
for p in $parts; do get $p; done
grep "  $t.part" PARTS.sha256 | sha256sum -c --quiet && echo "$(echo $parts | wc -w) parts OK"
[ -s $t ] || cat $parts > $t
sha256sum -c $t.sha256 && rm -f $parts
tar -xf $t -C $W/cohort && rm -f $t
cd $W/cohort
col=$(head -1 $W/panvram/MANIFEST.tsv | tr '\t' '\n' | grep -n "^${DATASET}_sha256$" | cut -d: -f1)
tail -n +2 $W/panvram/MANIFEST.tsv | awk -F'\t' -v c=$col -v d=$DATASET '{print $c "  " $1 "." d ".rr3"}' > want.sha256
sha256sum -c --quiet want.sha256 && echo "$(wc -l < want.sha256) archives, SHA-256 == MANIFEST.tsv"


In [ ]:
%%bash
# 5. On the GPU: open, sample 1024 x 8192, 1000 fetches (both against the CPU decoder), full decode of every assembly
#    with the FASTA XXH3 against the manifest; one block of C558 lines per resident form; log in $W/panvram_quickstart.txt
#    (and a copy next to the dataset source when it is a local/Drive directory)
set -euo pipefail
ulimit -c 0
cd $W/panvram
for form in $FORMS; do
  PANVRAM_PACKED_REF=${form%,*} PANVRAM_COMPACT_BLOCKS=${form#*,} python scripts/cohort558.py $W/cohort MANIFEST.tsv $DATASET
done 2>&1 | tee $W/panvram_quickstart.txt | grep -v "^C558 full	"
[[ "$DATASET_BASE_URL" == http* ]] || cp $W/panvram_quickstart.txt "$DATASET_BASE_URL/../quickstart_$(date -u +%Y-%m-%dT%H%MZ).txt" 2>/dev/null || true
